In [2]:
!nvcc --version

In [ ]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))


In [ ]:
%%writefile hello.cu
#include <stdio.h>

__global__ void helloKernel() {
  printf("hello from thread %d, block %d\n", threadIdx.x, blockIdx.x);
}

// general structure for a kernel:
// -> grid (1 per kernel)
//   -> blocks (65535 per dimension)
//   -> threads (1024 per block)
//   -> streaming multiprocessors(SM) (48)
//   -> warp size (32 threads per SM)
float *d_a;

// Implement a "kernel" (GPU function) that adds 10 to each position of vector
// `a` and stores it in vector `out`. 1 thread per position.
// --------------
//
//
__device__ void printThread() { printf("thread: %d\n", threadIdx.x); }
__global__ void addTen(float *a, float *out, int n) {
  printThread();
  out[threadIdx.x] = a[threadIdx.x] + 10;
}

int main() {
  float *d_a;
  float *d_out;
  float a[] = {1, 2, 3, 4, 5, 6, 7, 8};
  float out[8];
  cudaMalloc(&d_a, 8 * sizeof(float));
  cudaMalloc(&d_out, 8 * sizeof(float));
  cudaMemcpy(d_a, a, 8 * sizeof(float), cudaMemcpyHostToDevice);
  addTen<<<1, 8>>>(d_a, d_out, 8);
  cudaMemcpy(out, d_out, 8 * sizeof(float), cudaMemcpyDeviceToHost);
  for (int i = 0; i < 8; i++) {
    printf("array element %d: %f\n", i, out[i]);
  }
  // helloKernel<<<2, 4>>>();
  // helloKernel<<<1028, 4>>>();
  cudaDeviceSynchronize();
  return 0;
}


Overwriting hello.cu


In [7]:
!nvcc hello.cu -o hello

In [8]:
!./hello

array element 0: 0.000000
array element 1: 0.000000
array element 2: 0.000000
array element 3: 0.000000
array element 4: 0.000000
array element 5: 0.000000
array element 6: 0.000000
array element 7: 0.000000
thread: 0
thread: 1
thread: 2
thread: 3
thread: 4
thread: 5
thread: 6
thread: 7
